In [ ]:
from collections import Counter
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


Using device: cuda


In [4]:
print("Loading corpus and rebuilding vocabulary...")

with open("text8", "r") as f:
    text = f.read().lower()
    tokens = text.split()

min_freq = 5
tokens = tokens[:1_000_000]

# Build vocabulary
word_freq = Counter(tokens)
vocab = [(w, c) for w, c in word_freq.items() if c >= min_freq]
vocab.sort(key=lambda x: (-x[1], x[0]))

# Add <UNK> token
unk_count = sum(c for w, c in word_freq.items() if c < min_freq)
vocab.append(('<UNK>', unk_count))
vocab_size = len(vocab)

# Create mappings
word_to_idx = {w: i for i, (w, _) in enumerate(vocab)}
idx_to_word = {i: w for i, (w, _) in enumerate(vocab)}

print(f"Vocabulary size: {vocab_size:,}")
print(f"Sample words: {list(word_to_idx.keys())[:10]}")

Loading corpus and rebuilding vocabulary...
Vocabulary size: 13,967
Sample words: ['the', 'of', 'and', 'one', 'in', 'a', 'to', 'zero', 'nine', 'is']


In [5]:
n_embd = 300

class SkipGramModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.vocab_size = vocab_size
        self.n_embd = n_embd
        self.input_embeddings  = nn.Embedding(vocab_size, n_embd)
        self.output_embeddings = nn.Embedding(vocab_size, n_embd)

    def forward(self, center_words, context_words=None):
        center_embeds = self.input_embeddings(center_words)
        logits = center_embeds @ self.output_embeddings.weight.T
        loss = None
        if context_words is not None:
            loss = F.cross_entropy(logits, context_words)
        return logits, loss

In [6]:
print("\nLoading trained model...")
model = SkipGramModel().to(device)
model.load_state_dict(torch.load("naive_skipgram_model.pth", map_location=device))
model.eval()
print("✓ Model loaded successfully!")


Loading trained model...
✓ Model loaded successfully!


In [7]:
embeddings = model.input_embeddings.weight.data.cpu()
embeddings = F.normalize(embeddings, p=2, dim=1)
print(f"Embedding matrix shape: {embeddings.shape}")

Embedding matrix shape: torch.Size([13967, 300])


In [8]:
def get_most_similar(word, k=10):
    """Find the k most similar words to a given word."""
    if word not in word_to_idx:
        print(f"❌ '{word}' not in vocabulary.")
        return None
    
    word_id = word_to_idx[word]
    word_vec = embeddings[word_id]
    
    print(f"\n{'='*60}")
    print(f"Most similar words to: '{word}'")
    print('='*60)
    
    similarities = embeddings @ word_vec
    top_k_vals, top_k_indices = torch.topk(similarities, k + 1)
    
    results = []
    for i in range(1, k + 1):
        similar_word_id = top_k_indices[i].item()
        similar_word = idx_to_word[similar_word_id]
        similarity_score = top_k_vals[i].item()
        results.append((similar_word, similarity_score))
        print(f"  {i:2d}. {similar_word:<20} (Score: {similarity_score:.4f})")
    
    return results


def get_analogy(a, b, c, k=5):
    """
    Find word d such that: a - b + c ≈ d
    Example: king - man + woman ≈ queen
    """
    for word in [a, b, c]:
        if word not in word_to_idx:
            print(f"❌ '{word}' not in vocabulary.")
            return None
    
    vec_a = embeddings[word_to_idx[a]]
    vec_b = embeddings[word_to_idx[b]]
    vec_c = embeddings[word_to_idx[c]]
    
    target_vec = vec_a - vec_b + vec_c
    similarities = embeddings @ target_vec
    
    print(f"\n{'='*60}")
    print(f"Analogy: '{a}' - '{b}' + '{c}' = ?")
    print('='*60)
    
    top_k_vals, top_k_indices = torch.topk(similarities, k + 3)
    
    results = []
    count = 0
    for i in range(k + 3):
        result_id = top_k_indices[i].item()
        result_word = idx_to_word[result_id]
        score = top_k_vals[i].item()
        
        if result_word not in [a, b, c]:
            count += 1
            results.append((result_word, score))
            print(f"  {count:2d}. {result_word:<20} (Score: {score:.4f})")
            if count >= k:
                break
    
    return results


def compute_similarity(word1, word2):
    """Compute cosine similarity between two words."""
    if word1 not in word_to_idx or word2 not in word_to_idx:
        print(f"❌ One or both words not in vocabulary.")
        return None
    
    vec1 = embeddings[word_to_idx[word1]]
    vec2 = embeddings[word_to_idx[word2]]
    similarity = (vec1 @ vec2).item()
    
    print(f"\nSimilarity between '{word1}' and '{word2}': {similarity:.4f}")
    return similarity


In [9]:
print("\n" + "="*60)
print("SIMILARITY TESTS")
print("="*60)

# Test various word similarities
with torch.no_grad():
    get_most_similar('king', k=10)
    get_most_similar('france', k=10)
    get_most_similar('three', k=10)
    get_most_similar('walking', k=10)
    get_most_similar('computer', k=10)

print("\n" + "="*60)
print("ANALOGY TESTS")
print("="*60)

# Test word analogies
with torch.no_grad():
    get_analogy('king', 'man', 'woman', k=5)      # → queen
    get_analogy('paris', 'france', 'rome', k=5)   # → italy
    get_analogy('walk', 'walking', 'swim', k=5)   # → swimming
    get_analogy('good', 'better', 'bad', k=5)     # → worse
    get_analogy('big', 'bigger', 'small', k=5)    # → smaller

print("\n" + "="*60)
print("PAIRWISE SIMILARITY TESTS")
print("="*60)

# Test specific word pairs
with torch.no_grad():
    compute_similarity('king', 'queen')
    compute_similarity('man', 'woman')
    compute_similarity('france', 'paris')
    compute_similarity('cat', 'dog')
    compute_similarity('happy', 'sad')



SIMILARITY TESTS

Most similar words to: 'king'
   1. son                  (Score: 0.3967)
   2. sigismund            (Score: 0.3949)
   3. duke                 (Score: 0.3770)
   4. castile              (Score: 0.3678)
   5. prince               (Score: 0.3677)
   6. iii                  (Score: 0.3646)
   7. medici               (Score: 0.3597)
   8. pedro                (Score: 0.3570)
   9. lfhild               (Score: 0.3544)
  10. afonso               (Score: 0.3530)

Most similar words to: 'france'
   1. provence             (Score: 0.3989)
   2. spain                (Score: 0.3788)
   3. flee                 (Score: 0.3618)
   4. castile              (Score: 0.3553)
   5. french               (Score: 0.3519)
   6. toulouse             (Score: 0.3514)
   7. madrid               (Score: 0.3470)
   8. <UNK>                (Score: 0.3424)
   9. afonso               (Score: 0.3406)
  10. frankish             (Score: 0.3389)

Most similar words to: 'three'
   1. four                

In [11]:
print("\n" + "="*60)
print("INTERACTIVE MODE")
print("="*60)
print("\nYou can now test custom queries:")
print("  • get_most_similar('your_word', k=10)")
print("  • get_analogy('word1', 'word2', 'word3', k=5)")
print("  • compute_similarity('word1', 'word2')")
print("\nExample:")
print("  get_most_similar('python')")
print("  get_analogy('england', 'london', 'france')")


INTERACTIVE MODE

You can now test custom queries:
  • get_most_similar('your_word', k=10)
  • get_analogy('word1', 'word2', 'word3', k=5)
  • compute_similarity('word1', 'word2')

Example:
  get_most_similar('python')
  get_analogy('england', 'london', 'france')
